In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient IV: how far may one update move?

## Where this session fits

> **The limitation we start from: every episode costs something to collect, and the policy gradient uses a batch of episodes for *one* update and then throws it away**

- **a second problem:** a large step on a noisy estimate can also ruin the policy
    - both problems ask the same thing: how far may one update move?

- **The question:** how can we take several updates from the same batch, and keep each one small enough to trust?
- **Position: Part B (policy gradient), session 4 of its 4 sessions (the last of Part B)**
    - B2 and B3 filled slot 1, the weight
    - this session fills **slot 2**, how far one update moves (slot 3 is where the baseline comes from)

<img src="images/partB_arc_b4.png" width=70%>

- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. say why a batch of episodes goes stale after one update, and what the probability ratio corrects
2. say what a surrogate objective is (a stand-in for the true goal) and why its gradient at ratio $1$ is the policy gradient
3. read the clipped objective, and say which samples it stops pushing
4. read the PPO outline and say which model does what
5. say how a trust region and clipping limit the movement, and name what a KL penalty adds (a preview: developed in Part C)


**Recall: the gradient with an advantage, from B2 and B3**

> **Slot 1 is settled: each score function is weighted by an advantage, estimated by the return minus a baseline, by the critic's TD error, or by GAE (B3).**

$$
\nabla_\theta J(\theta) = \Exp{\traj \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) \; \advseq_{\traj,\tt} \right]
$$
- the **score function** $\nabla_\theta \log \pi_\theta(\act \mid \state)$ is the gradient of the log-probability of the action taken
    - it is the direction in parameter space that makes that action more likely
- the critic's TD error: $\delta_\tt = \rewseq_{\tt+1} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1}) - \statevalfun_\phi(\stateseq_\tt)$, with $\advseq_\tt \approx \delta_\tt$
- this session asks how far one update may move $\theta$ using this estimate

# Step 1 of 4: the limitation, a batch goes stale after one update

**Why this section?** B2 and B3 made the weight good. The next cost is how much each batch of episodes buys us.

> **A batch of episodes is expensive and goes stale after one update. The fix has two parts: correct for data from the old policy (step 2, the probability ratio), and limit how far $\theta$ moves (steps 3 and 4).**

<img src="images/b4_stale_batch.png" width=60%>

- the policy gradient is **on-policy**: the episodes must come from the policy being improved (defined in A4)
    - data from any other policy is **off-policy**
- one update changes $\theta$, so the batch now came from the **old policy** $\pi_{\theta_{\text{old}}}$
    - the gradient is an average under the *current* policy, which the old batch no longer samples
    - so far (REINFORCE in B2, actor-critic in B3) we sample, update once, and discard
    - PPO is also an actor-critic method, with B3's critic and advantage
    - what it adds is a way to update more than once
- **what makes an episode expensive** is more than its length
    - a robot's time and wear, a simulator run, or many tokens from a large network
    - scoring it (a reward may need a person to judge), and a noisy estimate needs many episodes


## Step 2 of 4: reusing old data, the probability ratio

**Why this section?** After the first update the batch is off-policy data: it came from the old policy, but the gradient needs the new policy's average. The ratio corrects this, so we can keep updating.

> **The probability ratio $r = \pi_\theta / \pi_{\theta_{\text{old}}}$ compares the new policy with the old one, and reweights old-policy data to give the new policy's average.**

- the **probability ratio**, for one state $\state$ and one action $\act$
    - $r_\theta(\state, \act) = \dfrac{\pi_\theta(\act \mid \state)}{\pi_{\theta_{\text{old}}}(\act \mid \state)}$
    - $r > 1$: the action became more likely, and $r < 1$: less likely
    - $r = 1$ at $\theta = \theta_{\text{old}}$
- in training the batch fixes the state and the action, so $r_\tt(\theta) = r_\theta(\state_\tt, \act_\tt)$ varies only with $\theta$
- as an **importance weight**: averaging over actions *sampled by the old policy*, times $r$, gives the average under the *new* policy

$$
\large \underbrace{\E_{\act \sim {\color{blue}{\pi_{\theta_{\text{old}}}}}(\cdot \mid \state)} \bigl[\, r_\theta(\state, \act)\, f(\act) \,\bigr]}_{\color{blue}{\text{actions sampled by the OLD policy}}} \;=\; \underbrace{\E_{\act \sim {\color{darkorange}{\pi_\theta}}(\cdot \mid \state)} \bigl[\, f(\act) \,\bigr]}_{\color{darkorange}{\text{the NEW policy's average}}}
$$

- $f$ is any function of the action, for example the advantage $\advseq$
    - here $\act$ is an action; do not read it as the advantage $\advseq$
- **a limit:** the ratio corrects the choice of *action* in each visited state, not *which states are visited*
    - so it is trustworthy only near $\theta_{\text{old}}$ (measured in this step)


**The ratio by numbers**

> **Goal: keep updating after the first update. For that we need the average advantage under the *new* policy, computed from actions sampled by the *old* policy.**

<img src="images/b3_ratio_bars.png" width=72%>

- the advantage of each action is measured against the old policy, so its old-policy average is zero
    - $0.5 \cdot (+1) + 0.3 \cdot (-1) + 0.2 \cdot (-1) = 0$
- **directly**, weighting by the new policy: $0.6 \cdot (+1) + 0.25 \cdot (-1) + 0.15 \cdot (-1) = 0.20$
- **from old samples**, weighting each by $r$: $0.5 \cdot 1.2 \cdot (+1) + 0.3 \cdot 0.833 \cdot (-1) + 0.2 \cdot 0.75 \cdot (-1) = 0.20$
    - the same answer: $r$ turns the old probability into the new one
    - **0.20 is the surrogate's improvement** of the new policy over the old one, in this one state
    - the surrogate averages such terms over all the steps of a batch (next slide)
- the identity needs the old policy to give positive probability to every action the new policy can take
- notation: this $r$ is the PPO paper's ratio, not a reward (written $\rew$ here)


## Step 2 of 4 (continued): the surrogate objective

**Why this section?** An objective we can optimize more than once on the same batch, whose gradient is still the policy gradient where it matters.

> **New term. The true goal $J(\theta)$ needs fresh episodes from the updated policy. A *surrogate* is a stand-in that reuses the old batch, so we can keep updating after the first update.**

- the ideal: after every update, recompute $J$ with new episodes from the updated policy
    - that throws away each expensive batch after one update
- the surrogate instead scores the updated policy on the *old* batch, reweighted by the ratio
    - $J_{\text{sur}}(\theta) = \E_\tt \bigl[\, r_\tt(\theta) \, \advseq_\tt \,\bigr]$, where $\E_\tt$ averages over the steps $\tt$ of the batch
    - the advantage is from B2, computed once with the old policy
- it estimates $J(\theta) - J(\theta_{\text{old}})$: the improvement of the updated policy over the old one

**How the surrogate differs from the true objective, and why "close" matters**

> **The surrogate is exact at the old policy. Away from it, it is only a stand-in, so we must keep the updated policy close to the old one.**

<img src="images/fl_surrogate_real.png" width=44%>

- the true improvement uses the new policy's actions in the states the *new* policy visits
- the surrogate uses the new policy's actions (through $r$), but in the states the *old* policy visited
    - the two agree only while the policies visit similar states
- at $\theta = \theta_{\text{old}}$ it is $0$, and its gradient is the policy gradient of B1, in the form we use
- **close** means a small average **KL divergence**: how different two distributions are, zero when they agree, averaged over the batch's states
- **FrozenLake, exact:** surrogate against true improvement
    - at an average KL of $0.02$: $0.011$ against $0.013$
    - at $0.8$: $0.06$ against $0.22$


**Why the gradient matches, and a sign convention**

- why the gradient matches: differentiate the ratio
    - only the numerator of $r_\tt$ depends on $\theta$, so $\pi_{\theta_{\text{old}}}$ stays outside the gradient
    - $\nabla_\theta r_\tt(\theta) = \dfrac{\nabla_\theta \pi_\theta(\actseq_\tt \mid \stateseq_\tt)}{\pi_{\theta_{\text{old}}}(\actseq_\tt \mid \stateseq_\tt)} = r_\tt(\theta) \, \nabla_\theta \log \pi_\theta(\actseq_\tt \mid \stateseq_\tt)$
    - the second step is the **log-derivative trick** (B1), $\nabla \pi = \pi \, \nabla \log \pi$, together with $\pi_\theta / \pi_{\theta_{\text{old}}} = r_\tt$
    - at $r_\tt = 1$ this is $\nabla_\theta \log \pi_\theta \; \advseq_\tt$, the term of the unified form
- **sign convention:** the surrogate is *maximized*, like $J$: a surrogate **objective**, as in the PPO paper
    - code usually minimizes, so it minimizes the negated surrogate, $-J_{\text{sur}}$
    - that negated version is often called the **surrogate loss**


## Step 3 of 4: keeping $\theta$ near $\theta_{\text{old}}$: three mechanisms

**Why this section?** The surrogate is trustworthy only near $\theta_{\text{old}}$. Methods differ in how they enforce that.

> **Three ways to keep $\theta$ near $\theta_{\text{old}}$: a trust region, clipping, and a KL penalty.**

<img src="images/b4_three_mechanisms.png" width=58%>

- all three limit **how far the policy moves**, and only clipping works through the ratio
    - the other two act on the KL divergence between whole distributions
    - clipping acts on the ratio of the *sampled* action, a cheaper stand-in
- **hard versus soft**
    - a **trust region** (TRPO, the older method) is a hard rule: any move inside is allowed, none outside
    - a **KL penalty** is soft: it subtracts the KL from the objective, so a move further out is allowed but costs more
    - **clipping** is neither: past the range, that sample's gradient is switched off (exactly zero)
- the trust region and clipping always limit the move from the **old policy**
- a KL penalty can also anchor to the old policy, or to a fixed **reference model**: RLHF-style PPO and GRPO do this (Part C)
    - then it limits drift over all of training, not over one update
    - RLHF is reinforcement learning from human feedback
- previews, not developed here: TRPO (Advanced course, RLLM-5), the reference model and the KL penalty (Part C)
- clipping has **no guarantee**: the ratio can still leave the range


# Step 3 of 4 (continued): PPO clips the ratio

**Why this section?** The clipped objective is the whole idea of PPO (proximal policy optimization): stop pushing a sample once its ratio has moved far enough.

> **PPO clips the ratio: once a sample's ratio has moved far enough in the direction its advantage favors, that sample is switched off: its gradient is exactly zero.**

- the **clipped surrogate** (maximized: the optimizer follows its gradient)

$$
J_{\text{clip}}(\theta) = \E_\tt \Bigl[\, \min{} \bigl( r_\tt(\theta)\, \advseq_\tt, \;\; \text{clip}\bigl(r_\tt(\theta),\, 1 - \epsilon_{\text{clip}},\, 1 + \epsilon_{\text{clip}}\bigr)\, \advseq_\tt \bigr) \,\Bigr]
$$

- $\text{clip}(x, a, b)$ limits $x$ to the interval $[a, b]$
- $\epsilon_{\text{clip}}$ is the clip range, typically $0.2$
    - the paper calls it $\epsilon$
    - we add the subscript to avoid the $\epsilon$ of $\epsilon$-greedy
- the `min` takes the more pessimistic of the plain and the clipped term, so a ratio that has gone too far earns no extra reward
- **hint:** where the clip applies, the gradient of $J_{\text{clip}}$ is **zero**, so that sample stops pushing
    - the picture on the next slide shows each case

**What clipping does, and does not do**

> **Clipping does not merely cap the ratio: past the range it replaces the ratio by a constant. The ratio drops out of that sample's term, so its gradient is exactly $0$.**
>
> In that case the `min` picks the clipped term $(1 \pm \epsilon_{\text{clip}})\,\advseq$, which contains no $\theta$.

<img src="images/ppo_clip.png" width=68%>

- with a positive advantage the objective stops rewarding $r$ above $1 + \epsilon_{\text{clip}}$, and with a negative one, below $1 - \epsilon_{\text{clip}}$
- flat means zero gradient: a **hard cutoff**, in one direction only, not a gradual vanishing gradient
- the sloped branches keep the gradient, so a mistake (a ratio that moved the wrong way) can be undone
- clipping does **not** bound the policy: other samples share the parameters and can still push a clipped sample's ratio further


**Reference: four samples, by numbers** ($\epsilon_{\text{clip}} = 0.2$, so the range is $[0.8, 1.2]$)

<img src="images/b4_ppo_clip_samples.png" width=86%>

- samples 1 and 4 have moved far enough in the direction their advantage asks: the gradient is **zero**
- samples 2 and 3 moved the wrong way: the gradient still acts, and pushes the ratio back
- inside the range $[0.8, 1.2]$ nothing is clipped, and the surrogate is the plain ratio objective


## Step 4 of 4: iterations and epochs, PPO in one loop

**Why this section?** Reuse is the goal of the whole session. The words must be exact, because supervised learning uses them differently.

> **An iteration generates a batch with the current policy; epochs are several passes over that fixed batch.**

<img src="images/b4_ppo_loop.png" width=76%>

- in supervised learning an epoch is one pass over fixed examples
    - here the examples are made by the *current* policy, so there are two nested loops
- at the end of the iteration $\theta_{\text{old}}$ is set to the new $\theta$
- implementations use mini-batches, and often add an entropy bonus (a reward for staying random)
- RLHF-style PPO adds a reference model and a KL term to this loop (C1)


**Reusing a batch: is it safe?**

- why this is safe enough
    - the first update is exactly on-policy: $r_\tt = 1$ and the gradient is the policy gradient
    - later epochs are slightly off-policy: the ratio corrects them, and clipping stops the samples that have moved far
    - implementations split a batch into mini-batches, so later mini-batches already have $r \neq 1$
- the number of epochs $K$ (often $3$ to $10$) is the dial
    - more epochs use the batch better, until the policy drifts far from the old one


**Reusing one batch, with and without the clip (FrozenLake)**

> **Reusing a small noisy batch for many epochs is a gamble. The clip turns it into a safe, small step.**

<img src="images/fl_ppo_epochs.png" width=64%>

- one batch of 32 episodes and 40 epochs
    - advantages from the exact state values, so the only noise is the batch of episodes
- $J(\theta)$ is exact after every epoch
- without the clip, some batches help a lot and others ruin the policy: the mean ends near $0$
- with the clip ($\epsilon_{\text{clip}} = 0.2$) the policy barely moves: safe, but it learns little from one batch
- the clip is the price of safety: it also limits what one batch can teach
    - this is the worst case for the clip: PPO draws a fresh batch every iteration, so each small safe step is repeated


**PPO in the unified form: what each part gives, and what it costs**

> **PPO is an actor-critic method: the weight and the baseline still come from B3's critic. What is new is a way to reuse a batch: the probability ratio, and the clip.**

<img src="images/b4_ppo_template.png" width=72%>

- **the new mechanism is the ratio**: new policy divided by old policy
    - the benefit: one batch serves several updates, not one
    - the catch: after the first update the policy is no longer the one that made the batch
    - so the ratio corrects the old data, and the clip keeps the policy close to the old one
- the clip is slot 2, how far one update moves


## The slot table: PPO changes only slot 2

**Why this section?** Placing PPO in the grid we have been filling, so the next method is a single change.

- B4 fills **slot 2**, how far one update moves
    - slots 1 and 3 are as in B2 and B3
    - slot 3 changes again in Part C

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Slot 1: weight $\advseq$</div></th><th nowrap><div align="left">Slot 2: how far</div></th><th nowrap><div align="left">Slot 3: baseline source</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">REINFORCE, over-simplified (B1)</div></td><td nowrap><div align="left">return, no baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">none</div></td></tr>
<tr><td nowrap><div align="left">REINFORCE with baseline (B2)</div></td><td nowrap><div align="left">return minus baseline</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">moving average</div></td></tr>
<tr><td nowrap><div align="left">actor-critic (B3), one update per batch</div></td><td nowrap><div align="left">TD error or GAE</div></td><td nowrap><div align="left">step size $\alpha$</div></td><td nowrap><div align="left">learned critic</div></td></tr>
<tr><td nowrap><div align="left"><b>PPO: actor-critic plus ratio and clip</b></div></td><td nowrap><div align="left">GAE with a learned critic</div></td><td nowrap><div align="left"><b>clipped ratio, $K$ epochs per batch</b></div></td><td nowrap><div align="left">learned critic</div></td></tr>
<tr><td nowrap><div align="left">GRPO</div></td><td nowrap><div align="left">group-relative advantage</div></td><td nowrap><div align="left">clipped ratio, as PPO (Part C)</div></td><td nowrap><div align="left">mean of a group of episodes</div></td></tr>
</tbody></table>

**Looking back: the on-policy versus off-policy thread, where PPO sits**

**Why this section?** The ratio is the policy-gradient answer to a question the value-based methods answered earlier: when may data from one policy train another?

> **Every update has a behavior policy (it generates the data) and a target policy (it is the one improved). Equal means on-policy, different means off-policy (A4).**

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Behavior policy</div></th><th nowrap><div align="left">Target policy</div></th><th nowrap><div align="left">Reuse of old data</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">SARSA</div></td><td nowrap><div align="left">its $\epsilon$-greedy policy</div></td><td nowrap><div align="left">the same policy</div></td><td nowrap><div align="left">no (on-policy)</div></td></tr>
<tr><td nowrap><div align="left">Q-learning</div></td><td nowrap><div align="left">$\epsilon$-greedy</div></td><td nowrap><div align="left">greedy</div></td><td nowrap><div align="left">yes, built into the update (replay)</div></td></tr>
<tr><td nowrap><div align="left">REINFORCE, actor-critic</div></td><td nowrap><div align="left">the current policy</div></td><td nowrap><div align="left">the same policy</div></td><td nowrap><div align="left">no: one update, then discard</div></td></tr>
<tr><td nowrap><div align="left"><b>PPO</b></div></td><td nowrap><div align="left"><b>the old policy</b></div></td><td nowrap><div align="left"><b>the new policy</b></div></td><td nowrap><div align="left"><b>yes, a few epochs, through the ratio and the clip</b></div></td></tr>
</tbody></table>

- Q-learning gets reuse from its update rule
- a policy gradient has to pay for reuse with a correction, the ratio
- PPO is only slightly off-policy: the clip keeps the target close to the behavior
- data from the old policy covers the states the old policy visits
    - the ratio corrects the choice of *action* in each visited state, not *which states are visited*
    - so PPO keeps the move small: the visited states stay nearly the same

# What we established, and what it sets up

- **Established**
    - **PPO limits how far one update moves, so a batch can be reused for several epochs**
    - the policy gradient is on-policy, so a batch goes stale after one update
    - reusing it needs a correction and a limit
    - the **probability ratio** is an importance weight: a re-weighting that lets old-policy samples estimate a new-policy average
    - the **surrogate** $\E[r_\tt \advseq_\tt]$ has the policy gradient as its gradient at ratio $1$
    - **PPO** clips the ratio, so samples that have moved far enough in the direction of their advantage stop pushing
        - this allows several epochs per batch
    - **TRPO**, the older method, pursues the same goal with a hard KL constraint (Advanced course, RLLM-5)
        - the penalty toward a frozen reference policy returns with language models (Part C)
    - PPO keeps B3's cost: a learned critic, a second large network
    - PPO is a controlled, slightly off-policy update
        - on-policy data matters because it covers the states the model will actually visit (Advanced course)

**What it sets up**

- **Sets up**
    - Part C: slot 3, do we need the critic at all?
        - sample several episodes from the same start and use their own average as the baseline (GRPO)
        - slot 2 keeps the clipped ratio
